In [ ]:
import base64
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Image, HTML

In [ ]:
load_dotenv("../.env") # load our env vars
client = OpenAI(
    base_url="https://api.meta.ai/v1",
    api_key=os.environ["MODEL_API_KEY"],
)

In [ ]:
# A nice way for use to display a table of images inline.
def display_image_table(data):
  html = '<table><tr>'
  for i, item in enumerate(data):
      # Supports both dictionaries and OpenAI SDK objects
      image = item["b64_json"] if isinstance(item, dict) else item.b64_json

      html += f'''
      <td style="padding: 8px; text-align: center;">
          <img src="data:image/webp;base64,{image}"
              style="width: 200px; height: auto; border-radius: 8px;">
          <p>Image {i + 1}</p>
      </td>
      '''
      # Start a new row every 3 images
      if (i + 1) % 3 == 0:
          html += '</tr><tr>'
  html += '</tr></table>'
  display(HTML(html))

In [ ]:
# https://dev.meta.ai/docs/api-reference/images/schemas#create-image-request

prompt="an image of canned lunch meat product, Canadian branded"
#prompt="generate a pack of cigerattes" # This should throw a 400 due to moderation

response = client.images.generate(
  model="muse-image-1.0",
  prompt=prompt,
  size="1024x1024",          # width x height
  output_format="png",       # webp (default), png, jpeg
  response_format="b64_json",# b64_json or url
  n=1,                       # of images, max 10
  # == despite being in the reference docs, not allowed to set
  # moderation can't be set and other options maybe not via the OpenAI SDK
  # temperature=1,             # 0-2 
  # reasoning_strength="high", # low, high 
  # moderation="low",          # low / auto
)

# Response Structure ===
# https://dev.meta.ai/docs/api-reference/images/schemas#images-response
# created: 1784584435           # Unix timestamp (seconds)
# data:
#   - b64_json: "UklGR..."    # Base64-encoded image data
#   # Additional entries if n > 1
# output_format: "webp"        # webp | png | jpeg
# background: "opaque"        # Always opaque
# usage:
#   input_tokens: 9831
#   output_tokens: 749
#   total_tokens: 10580

In [ ]:
display_image_table(response.data)

In [ ]:
# https://dev.meta.ai/docs/api-reference/images/schemas#create-image-edit-request

# we'll just edit on the first image result
image_bytes = base64.b64decode(response.data[0].b64_json)
response_edit = client.images.edit(
    model="muse-image-1.0",
    prompt="Change the product's name to Oinker's, make the labeling look like an expensive brand, should be on a sliver platter with fine-dining background. Use the pig mascot in the provided reference image",
    image=("input.png", image_bytes, "image/png"),
    n=3,
)
# Response Structure ===
# same as generate.

In [ ]:
display_image_table(response_edit.data)

In [ ]:
# OpenAI SDK does not support all the parameters of Meta's AI SDK
# so we need to make direct requests to the API when we want to supply
# multiple images.

# First image: previous Muse generation
generated_image = response.data[0].b64_json

# Second image: local mascot reference
mascot_image = base64.b64encode(
    Path("oinkers.png").read_bytes()
).decode("utf-8")

api_response = requests.post(
  "https://api.meta.ai/v1/images/edits",
  headers={"Authorization": f"Bearer {os.environ['MODEL_API_KEY']}"},
  json={
    "model": "muse-image-1.0",
    "prompt": (
      "Change the product's name to Oinker's. "
      "Make the labeling look like an expensive brand. "
      "Place it on a silver platter in a fine-dining setting. "
      "Use the pig mascot from the second reference image cover half the label."
    ),
    "size": "1024x1024",
    "images": [ # 1 to 10 reference images
      {"image_url": f"data:image/{response.output_format or 'webp'};base64,{generated_image}"},
      {"image_url": f"data:image/png;base64,{mascot_image}"}
    ],
    "n": 1
  },
  timeout=180
)

api_response.raise_for_status()

In [ ]:
response_api_edit = api_response.json()
display_image_table(response_api_edit['data'])